# mv-PIKN: 3D latent plots
Run all cells in Colab. Uses the existing Koopman latent HDF5 exports, with no model retraining. PCA is fitted on 50,000 training-ready windows (including label −1); the same projector is applied to validation. PC3 cannot be reconstructed from the previous two-coordinate CSVs.

Outputs: interactive HTML and static PNG/PDF plots by class, by validation subject, and for the previously selected event. Original exports are preserved. These are inner-validation visualizations, not evidence of physiological identification or test generalization.

In [ ]:
%pip -q install h5py numpy pandas scikit-learn matplotlib plotly
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
from pathlib import Path
import json, shutil, tempfile
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA
from datetime import datetime, timezone

RUN = Path('/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z')
EXPORT_DIR = ''  # Optional: exact folder containing koopman_train_latents.h5
SEED = 20261101
VARIANT = 'koopman'
EXPECTED_HASH = '3f0bb19589912f97aaf9f9cae05c5aa894e41ebb51fef043fd2cc8040ce5635f'
if EXPORT_DIR:
    src = Path(EXPORT_DIR)
else:
    candidates=[]
    for m in (RUN/'latent_exports_v1').rglob('manifest.json'):
        d=json.loads(m.read_text())
        if (d.get('scope')=='inner_01' and d.get('seed')==20261028
            and d.get('provenance',{}).get(VARIANT,{}).get('model_checkpoint_sha256')==EXPECTED_HASH
            and (m.parent/f'{VARIANT}_train_latents.h5').exists()
            and (m.parent/f'{VARIANT}_validation_latents.h5').exists()):
            candidates.append(m.parent)
    if not candidates: raise FileNotFoundError('Set EXPORT_DIR to the original latent export folder containing both HDF5 files.')
    src=sorted(candidates)[-1]
print('Reading:',src)
work=Path(tempfile.mkdtemp(prefix='latent_3d_'))
paths={}
for role in ['train','validation']:
    origin=src/f'{VARIANT}_{role}_latents.h5'
    with h5py.File(origin,'r') as f:
        assert f.attrs['status']=='complete' and f.attrs['role']==role
        assert f.attrs['checkpoint_sha256']==EXPECTED_HASH, 'Unexpected checkpoint'
        assert f.attrs['scope']=='inner_01' and f.attrs['outer']=='outer_chb01'
    paths[role]=work/origin.name
    shutil.copy2(origin,paths[role])
out=work/'figures';out.mkdir()


In [ ]:
def sample_export(path, limit=50000, seed=20261101):
    """Uniform sample over exported ready windows using smallest random keys.
    Includes label -1 in the training PCA population. Bounded RAM.
    """
    rng = np.random.default_rng(seed);keys = np.empty(0);z = np.empty((0,32),dtype='f4');meta = pd.DataFrame()
    with h5py.File(path,'r') as f:
        if f.attrs['status'] != 'complete': raise ValueError('Incomplete latent export')
        role = f.attrs['role']
        for name in sorted(f):
            g = f[name]
            for start in range(0,len(g['label']),2048):
                stop = min(start+2048,len(g['label']));keep = g['prediction_ready'][start:stop].astype(bool)
                if not keep.any():continue
                zz = g['z'][start:stop][keep]
                if not np.isfinite(zz).all():raise ValueError('Missing exported ready latent')
                mm = pd.DataFrame(dict(subject=g.attrs['subject'],case=g.attrs['case'],file=g.attrs['relative_path'],role=role,
                                       index=g['index'][start:stop][keep],label=g['label'][start:stop][keep],
                                       decision_s=g['decision_s'][start:stop][keep]))
                keys = np.r_[keys,rng.random(len(zz))];z = np.concatenate([z,zz]);meta = pd.concat([meta,mm],ignore_index=True)
                if len(keys)>limit:
                    ids = np.argpartition(keys,limit-1)[:limit]
                    keys=keys[ids];z=z[ids];meta=meta.iloc[ids].reset_index(drop=True)
    if len(z)<3:raise ValueError('Insufficient exported windows for PCA')
    return z,meta



z, fit_meta = sample_export(paths['train'],50000,SEED)
if not np.any(np.ptp(z,axis=0)>0): raise ValueError('Constant training representation')
pca=PCA(n_components=3,svd_solver='full').fit(z)
assert np.isfinite(pca.explained_variance_ratio_).all()
np.savez_compressed(out/'koopman_training_pca_3d.npz',mean=pca.mean_,components=pca.components_,explained_variance_ratio=pca.explained_variance_ratio_)
frames=[]
for role in ['train','validation']:
    zz,meta=sample_export(paths[role],12000,SEED)
    xyz=pca.transform(zz)
    for j in range(3):meta[f'PC{j+1}']=xyz[:,j]
    meta.to_csv(out/f'koopman_{role}_pca_3d_points.csv',index=False)
    frames.append(meta)
points=pd.concat(frames,ignore_index=True)
axes={f'PC{j+1}':f'PC{j+1} ({v:.1%})' for j,v in enumerate(pca.explained_variance_ratio_)}
print('Training variance explained:',pca.explained_variance_ratio_, 'Total:',pca.explained_variance_ratio_.sum())


In [ ]:
class_names={0:'Interictal',1:'Preictal',2:'Ictal'}
colors={'Interictal':'#2878B5','Preictal':'#E69F00','Ictal':'#CC3377'}
limits={k:(points[k].min(),points[k].max()) for k in axes}
def finish(fig,stem):
    fig.update_layout(template='plotly_white',scene=dict(xaxis_title=axes['PC1'],yaxis_title=axes['PC2'],zaxis_title=axes['PC3']))
    fig.write_html(out/f'{stem}.html',include_plotlyjs=True)
    fig.show()
def static_scatter(data,color_col,palette,title,stem):
    fig=plt.figure(figsize=(9,7));ax=fig.add_subplot(111,projection='3d')
    for key,g in data.groupby(color_col,sort=True):
        ax.scatter(g.PC1,g.PC2,g.PC3,s=5,alpha=.4,color=palette[key],label=f'{key} (n={len(g):,})',rasterized=True)
    ax.set(xlabel=axes['PC1'],ylabel=axes['PC2'],zlabel=axes['PC3'],title=title,
           xlim=limits['PC1'],ylim=limits['PC2'],zlim=limits['PC3'])
    ax.view_init(elev=22,azim=40);ax.legend(fontsize=8,loc='upper left')
    fig.tight_layout()
    for ext in ['png','pdf']:fig.savefig(out/f'{stem}.{ext}',dpi=300,bbox_inches='tight')
    plt.show();plt.close(fig)
for role in ['train','validation']:
    d=points[(points.role==role)&points.label.isin(class_names)].copy()
    d['class']=d.label.map(class_names)
    title=f'Koopman / {role}: current fused latent — training-fitted PCA'
    fig=px.scatter_3d(d,x='PC1',y='PC2',z='PC3',color='class',color_discrete_map=colors,opacity=.45,
                      hover_data=['subject','file','index','decision_s'],title=title)
    fig.update_traces(marker_size=2)
    fig.update_layout(scene=dict(xaxis_range=limits['PC1'],yaxis_range=limits['PC2'],zaxis_range=limits['PC3']))
    finish(fig,f'koopman_{role}_3d_by_class')
    static_scatter(d,'class',colors,title,f'koopman_{role}_3d_by_class')
d=points[points.role=='validation'].copy()
subjects=sorted(d.subject.unique());subject_colors = [
    "#88CCEE", "#CC6677", "#DDCC77",
    "#117733", "#332288", "#AA4499",
]
palette = dict(zip(subjects, subject_colors))
fig=px.scatter_3d(d,x='PC1',y='PC2',z='PC3',color='subject',color_discrete_map=palette,opacity=.4,
                  hover_data=['label','file','index'],title='Validation subjects (including label −1)')
fig.update_traces(marker_size=2);finish(fig,'koopman_validation_3d_by_subject')
static_scatter(d,'subject',palette,'Validation subjects (including label −1)','koopman_validation_3d_by_subject')


In [ ]:
# Reuse the EXACT event and rows selected in the earlier export.
trajectory_path=src/'koopman_event_trajectory.csv'
if trajectory_path.exists():
    t=pd.read_csv(trajectory_path)
    xyz=np.full((len(t),3),np.nan)
    with h5py.File(paths['validation'],'r') as f:
        for g in f.values():
            ids=np.flatnonzero(t.file.to_numpy()==g.attrs['relative_path'])
            if not len(ids):continue
            times=g['decision_s'][:]
            if len(np.unique(times))!=len(times):raise ValueError('Duplicate timestamps within EDF')
            positions=pd.Index(times).get_indexer(t.iloc[ids].decision_s)
            if (positions<0).any():raise ValueError('Trajectory row missing from HDF5')
            zz=g['z'][:][positions]
            good=np.isfinite(zz).all(axis=1)
            xyz[ids[good]]=pca.transform(zz[good])
    for j in range(3):t[f'PC{j+1}']=xyz[:,j]
    t.to_csv(out/'koopman_event_trajectory_3d.csv',index=False)
    good=np.isfinite(xyz).all(axis=1)
    fig=go.Figure(go.Scatter3d(x=t.PC1,y=t.PC2,z=t.PC3,mode='markers',
        marker=dict(size=3,color=t.minutes_from_onset,colorscale='Viridis',colorbar=dict(title='Minutes from onset')),
        text=t.file,customdata=t.minutes_from_onset,hovertemplate='%{text}<br>%{customdata:.2f} min<extra></extra>'))
    fig2=plt.figure(figsize=(9,7));ax=fig2.add_subplot(111,projection='3d')
    sc=ax.scatter(t.PC1,t.PC2,t.PC3,c=t.minutes_from_onset,cmap='viridis',s=7)
    # Break connections at every EDF boundary, unavailable row or nonconsecutive 4-second step.
    for i in range(1,len(t)):
        if good[i-1] and good[i] and t.file.iloc[i]==t.file.iloc[i-1] and np.isclose(t.decision_s.iloc[i]-t.decision_s.iloc[i-1],4):
            pair=xyz[i-1:i+1]
            fig.add_trace(go.Scatter3d(x=pair[:,0],y=pair[:,1],z=pair[:,2],mode='lines',line=dict(color='gray',width=1),showlegend=False,hoverinfo='skip'))
            ax.plot(*pair.T,color='gray',alpha=.25,linewidth=.4)
    fig.update_layout(title='Validation event trajectory — same event as the 2D export')
    finish(fig,'koopman_event_trajectory_3d')
    ax.set(xlabel=axes['PC1'],ylabel=axes['PC2'],zlabel=axes['PC3'],title='Validation event trajectory')
    fig2.colorbar(sc,ax=ax,shrink=.6,label='Minutes from onset')
    for ext in ['png','pdf']:fig2.savefig(out/f'koopman_event_trajectory_3d.{ext}',dpi=300,bbox_inches='tight')
    plt.show();plt.close(fig2)
else:
    print('No earlier event trajectory CSV found; class and subject plots are complete.')


In [ ]:
report=dict(source_export=str(src),checkpoint_sha256=EXPECTED_HASH,pca_fit_role='train',fit_windows=len(z),
    seed=SEED,explained_variance_ratio=pca.explained_variance_ratio_.tolist(),outer_test_accessed=False,
    representation='current fused z',plot_windows_per_role=12000)
(out/'pca_3d_report.json').write_text(json.dumps(report,indent=2))
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
destination=src/f'latent_3d_{stamp}'
shutil.copytree(out,destination)
archive=shutil.make_archive(str(work/'latent_3d_figures'),'zip',out)
shutil.copy2(archive,destination/'latent_3d_figures.zip')
print('Saved:',destination)
print('Open HTML files for rotation/zoom. PNG/PDF files are static paper figures.')
